In [ ]:
s_e1=s[s['Experiment_ID']=='e1']
s_e2=s[s['Experiment_ID']=='e2']
s_e3=s[s['Experiment_ID']=='e3']
u_e1=u[u['Experiment_ID']=='e1']
u_e2=u[u['Experiment_ID']=='e2']
u_e3=u[u['Experiment_ID']=='e3']
c_e1=c[c['Experiment_ID']=='e1']
c_e2=c[c['Experiment_ID']=='e2']
c_e3=c[c['Experiment_ID']=='e3']

In [ ]:
# [W1] 配置 + 重新定义 Wasserstein distance + 读取 PD
import os, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import homcloud.interface as hc
from joblib import Parallel, delayed
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ================= 0. 配置 =================
MODE        = 'superlevel'   # 修改：原来是 'sublevel'；必须与生成 .pdgm 时一致
DIM         = 0               # 同调维数
THRESHOLD   = 0.01            # persistence 阈值（三组、三次实验必须相同）
TOPN        = 2000            # None=保留全部点；需在 Methods 说明
ORDER       = 1               # Wasserstein 阶数 q
INTERNAL_P  = 2               # 地面距离：2=欧氏，np.inf=L∞
BACKEND     = 'pot'           # 'pot' = gudhi.wasserstein（精确 OT，慢）；'hera' = gudhi.hera（快，结果一致到 delta 精度）
FORCE_RECOMPUTE = True        # 第一次换成新 distance 时设 True，确保不读旧缓存
N_PERM      = 10000
N_BOOT      = 5000

n           = 10
experiments = ['e1', 'e2', 'e3']
groups      = ['SNCA', 'UBL3', 'CAAX']                         # 标签 0,1,2
region_of   = {'SNCA': 'SNCA', 'UBL3': 'UBL3', 'CAAX': 'CAAX'}
colors      = {'SNCA': '#264a5f', 'UBL3': '#d15034', 'CAAX': '#f1ae2d'}

dfs = {
    'e1': {'SNCA': s_e1, 'UBL3': u_e1, 'CAAX': c_e1},
    'e2': {'SNCA': s_e2, 'UBL3': u_e2, 'CAAX': c_e2},
    'e3': {'SNCA': s_e3, 'UBL3': u_e3, 'CAAX': c_e3},
}
# 修改：删除原来的 pd_dir 定义（指向 pd/，会覆盖 Cell 9 里 superlevel 的 pd_super/），沿用 Cell 9 的 pd_dir
paths  = {e: pd_dir for e in experiments}
out_dir = f'/Users/yolanda/Desktop/cell topology data/{direct}/result_mmCen/wasserstein_out/'
os.makedirs(out_dir, exist_ok=True)

INT_TAG = 'inf' if np.isinf(INTERNAL_P) else str(INTERNAL_P)
TAG = f"{MODE}_H{DIM}_thr{THRESHOLD}_top{TOPN}_W{ORDER}_int{INT_TAG}_{BACKEND}"   # 所有输出文件都带这个标签
print('Run tag:', TAG)

# ================= 1. Wasserstein distance（方向感知）=================
def _check_orientation(x, mode):
    if mode == 'superlevel':
        if np.any(x[:, 0] <= x[:, 1]):
            raise ValueError("Expected superlevel intervals with birth > death. Check PD convention.")
    elif mode == 'sublevel':
        if np.any(x[:, 0] >= x[:, 1]):
            raise ValueError("Expected sublevel intervals with birth < death. Check PD convention.")
    else:
        raise ValueError(f"Unknown mode: {mode}")

def wasserstein_distance(pd1, pd2, order=ORDER, internal_p=INTERNAL_P, mode=MODE, backend=BACKEND):
    """
    pd1, pd2: HomCloud PD 的原始 (birth, death) 坐标，N×2。
    superlevel (birth > death) 先取负转成 birth < death 再交给 GUDHI；
    坐标整体取负是等距变换，不改变 Wasserstein 距离，只是满足 GUDHI 的输入约定。
    """
    a = np.asarray(pd1, dtype=float).reshape(-1, 2)
    b = np.asarray(pd2, dtype=float).reshape(-1, 2)
    if len(a) == 0 or len(b) == 0:
        raise ValueError("Empty persistence diagram: cannot compare without an explicit empty-PD policy.")
    if not (np.all(np.isfinite(a)) and np.all(np.isfinite(b))):
        raise ValueError("PD contains non-finite coordinates; filter essential intervals first.")
    _check_orientation(a, mode); _check_orientation(b, mode)
    if mode == 'superlevel':
        a, b = -a, -b
    if backend == 'hera':
        import gudhi.hera as gh
        return gh.wasserstein_distance(a, b, order=order, internal_p=internal_p)
    import gudhi.wasserstein as gw
    return gw.wasserstein_distance(a, b, order=order, internal_p=internal_p)

# ================= 2. 读取 PD（保持 HomCloud 原始方向）=================
def load_pd(save_path, samplename, region, mode=MODE, thr=THRESHOLD, topn=TOPN, dim=DIM):
    pd_obj = hc.PDList(save_path + samplename + "grayscale" + region + ".pdgm").dth_diagram(dim)
    b = np.asarray(pd_obj.births, float); d = np.asarray(pd_obj.deaths, float)
    ok = np.isfinite(b) & np.isfinite(d)          # 去掉 essential（无穷）区间
    b, d = b[ok], d[ok]
    pers = np.abs(d - b)
    keep = pers > thr                              # 同时去掉 birth == death 的零长度点
    arr = np.column_stack([b[keep], d[keep]])
    pers = pers[keep]
    if topn is not None and len(arr) > topn:
        arr = arr[np.argsort(-pers)[:topn]]
    if len(arr) == 0:
        raise ValueError(f"{samplename} {region}: 过滤后没有点")
    _check_orientation(arr, mode)                  # 方向与 MODE 不符会在这里就报错
    return arr

pds_all, cell_ids = {}, {}
for e in experiments:
    lst, ids = [], []
    for g in groups:                               # 顺序固定：SNCA, UBL3, CAAX
        labs = list(dfs[e][g]['label1'].unique())
        assert len(labs) == n, f"{e} {g}: 细胞数 {len(labs)} != {n}"
        for lab in labs:
            lst.append(load_pd(paths[e], lab, region_of[g])); ids.append(f"{e}_{g}_{lab}")
    pds_all[e], cell_ids[e] = lst, ids
    print(e, '每个细胞的点数范围:', min(map(len, lst)), '-', max(map(len, lst)))

# 快速自检：同一个 PD 距离应为 0，对称性成立
P0 = pds_all['e1']
assert np.isclose(wasserstein_distance(P0[0], P0[0]), 0)
assert np.isclose(wasserstein_distance(P0[0], P0[1]), wasserstein_distance(P0[1], P0[0]))
print('distance 自检通过')



In [ ]:
# [W2] 每次实验一个 30×30 距离矩阵（并行 + 带参数标签的缓存）
def dist_matrix(P):
    N = len(P); pairs_ij = list(itertools.combinations(range(N), 2))
    vals = Parallel(n_jobs=-1)(delayed(wasserstein_distance)(P[i], P[j]) for i, j in pairs_ij)
    M = np.zeros((N, N))
    for (i, j), v in zip(pairs_ij, vals):
        M[i, j] = M[j, i] = v
    return M

D = {}
for e in experiments:
    f = out_dir + f"D_{TAG}_{e}.npy"
    if os.path.exists(f) and not FORCE_RECOMPUTE:
        D[e] = np.load(f); print(e, '读取缓存', f)
    else:
        D[e] = dist_matrix(pds_all[e]); np.save(f, D[e]); print(e, '已计算并保存', f)
    assert np.allclose(D[e], D[e].T) and np.allclose(np.diag(D[e]), 0)
    pd.DataFrame(D[e], index=cell_ids[e], columns=cell_ids[e]).to_csv(out_dir + f"D_{TAG}_{e}.csv")

lab_full = np.repeat([0, 1, 2], n)      # 0=SNCA, 1=UBL3, 2=CAAX
pairs = {'UBL3 vs SNCA': (1, 0), 'UBL3 vs CAAX': (1, 2), 'CAAX vs SNCA': (2, 0)}
names = list(pairs)



In [ ]:
# [W3] 主分析：分层置换检验（均值；每次实验内部打乱标签）
def prepare(pair):
    a, b = pair; out = []
    for e in experiments:
        idx = np.where((lab_full == a) | (lab_full == b))[0]
        iu = np.triu_indices(len(idx), 1)
        out.append((D[e][np.ix_(idx, idx)][iu], iu, lab_full[idx]))
    return out

def diff_ratio_one(d, iu, lab):
    same = lab[iu[0]] == lab[iu[1]]
    w, bw = d[same].mean(), d[~same].mean()
    return bw - w, bw / w, w, bw

def strat_perm_test(pair, n_perm=N_PERM, seed=0):
    rng = np.random.default_rng(seed)
    blocks = prepare(pair)
    obs = [diff_ratio_one(*blk) for blk in blocks]
    diffs   = np.array([o[0] for o in obs]); ratios  = np.array([o[1] for o in obs])
    within  = np.array([o[2] for o in obs]); between = np.array([o[3] for o in obs])
    obs_stat = diffs.mean()
    null = np.empty(n_perm)
    for k in range(n_perm):
        null[k] = np.mean([diff_ratio_one(d, iu, rng.permutation(lab))[0] for d, iu, lab in blocks])
    p = (1 + np.sum(null >= obs_stat)) / (n_perm + 1)       # 单侧：between > within
    return dict(diffs=diffs, ratios=ratios, within=within, between=between,
                diff_mean=obs_stat, ratio_mean=ratios.mean(), n_pos=int((diffs > 0).sum()), p=p)

res = {k: strat_perm_test(v) for k, v in pairs.items()}
p_adj = multipletests([res[k]['p'] for k in names], method='holm')[1]

summary = pd.DataFrame([{
    'comparison': k,
    'ratio_e1/e2/e3': np.round(res[k]['ratios'], 3),
    'diff_e1/e2/e3': np.round(res[k]['diffs'], 2),
    'consistent': f"{res[k]['n_pos']}/3",
    'mean_diff': res[k]['diff_mean'], 'mean_ratio': res[k]['ratio_mean'],
    'p_raw': res[k]['p'], 'p_holm': pa} for k, pa in zip(names, p_adj)])
print('=== 主分析：cell-level permutation stratified by experiment (mean) ===')
print(summary.to_string(index=False))
summary.to_csv(out_dir + f"wasserstein_stats_mean_{TAG}.csv", index=False)



In [ ]:
# [W4] 图：热图 / 组内 vs 组间分布 / 每次实验的比值
# 图 1：三次实验热图（统一色标）
vmax = max(D[e].max() for e in experiments)
fig, axes = plt.subplots(1, 3, figsize=(17, 5.4))
for ax, e in zip(axes, experiments):
    sns.heatmap(D[e], cmap="coolwarm", vmin=0, vmax=vmax, ax=ax,
                xticklabels=False, yticklabels=False,
                cbar_kws={"label": "Wasserstein Distance"})
    ax.invert_yaxis()
    for i in [n, 2*n]:
        ax.axhline(i, color='white', lw=2); ax.axvline(i, color='white', lw=2)
    for i, g in enumerate(groups):
        ax.text(i*n + n/2, -1.0, g, ha='center', va='top', fontsize=11)
    ax.set_title(e, fontsize=14)
plt.tight_layout()
plt.savefig(out_dir + f"Wasserstein_heatmap_{TAG}.png", dpi=300); plt.show()

# 图 2：组内 vs 组间距离分布
rec = []
for e in experiments:
    for name, (a, b) in pairs.items():
        idx = np.where((lab_full == a) | (lab_full == b))[0]
        iu = np.triu_indices(len(idx), 1)
        d = D[e][np.ix_(idx, idx)][iu]; l = lab_full[idx]
        same = l[iu[0]] == l[iu[1]]
        rec += [dict(experiment=e, comparison=name, type='Within-group',  distance=v) for v in d[same]]
        rec += [dict(experiment=e, comparison=name, type='Between-group', distance=v) for v in d[~same]]
df_dist = pd.DataFrame(rec)
df_dist.to_csv(out_dir + f"wasserstein_within_between_{TAG}.csv", index=False)

g = sns.catplot(data=df_dist, x='comparison', y='distance', hue='type', col='experiment',
                kind='violin', split=True, inner='quartile', cut=0, linewidth=1,
                palette={'Within-group': '#9DB4D8', 'Between-group': '#3E5C99'},
                height=4, aspect=0.95)
g.set_xlabels(''); g.set_ylabels('Wasserstein Distance')
for ax in g.axes.flat: ax.tick_params(axis='x', rotation=20)
g.savefig(out_dir + f"Wasserstein_violin_{TAG}.png", dpi=300); plt.show()

# 图 3：每次实验的 between/within 比值（可重复性）
fig, ax = plt.subplots(figsize=(5.5, 4.5))
mk = {'e1': 'o', 'e2': 's', 'e3': '^'}
for i, (k, pa) in enumerate(zip(names, p_adj)):
    for j, e in enumerate(experiments):
        ax.scatter(i + (j - 1) * 0.12, res[k]['ratios'][j], marker=mk[e], s=60,
                   color='#3E5C99', zorder=3, label=e if i == 0 else None)
    ax.hlines(res[k]['ratios'].mean(), i - 0.25, i + 0.25, color='k', lw=2)
    star = '***' if pa < .001 else '**' if pa < .01 else '*' if pa < .05 else 'n.s.'
    ax.text(i, res[k]['ratios'].max() * 1.02, star, ha='center', va='bottom', fontsize=13)
ax.axhline(1, color='gray', ls='--')
ax.set_xticks(range(3)); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel('Between / within distance ratio')
ymax = max(res[k]['ratios'].max() for k in names)
ax.set_ylim(min(0.95, min(res[k]['ratios'].min() for k in names) * 0.98), ymax * 1.12)
ax.legend(title='Experiment', frameon=False, loc='upper right', ncol=3)
plt.tight_layout(); plt.savefig(out_dir + f"Wasserstein_ratio_{TAG}.png", dpi=300); plt.show()



In [ ]:
# [W5] 稳健性分析：中位数 + 分层置换 + 分层 bootstrap 95% CI
STAT = np.median
STAT_NAME = 'median' if STAT is np.median else 'mean'

def pair_blocks(pair):
    a, b = pair; out = []
    for e in experiments:
        idx = np.where((lab_full == a) | (lab_full == b))[0]
        out.append((D[e][np.ix_(idx, idx)], lab_full[idx]))
    return out

def within_between(Dsub, lab, ids=None):
    iu = np.triu_indices(len(lab), 1)
    d = Dsub[iu]; same = lab[iu[0]] == lab[iu[1]]
    if ids is not None:                              # 剔除 bootstrap 中同一细胞的自身配对
        keep = ids[iu[0]] != ids[iu[1]]
        d, same = d[keep], same[keep]
    return d[same], d[~same]

def pooled_stat(blocks, labs=None):
    W, B = [], []
    for k, (Dsub, lab) in enumerate(blocks):
        w, bt = within_between(Dsub, lab if labs is None else labs[k])
        W.append(w); B.append(bt)
    W, B = np.concatenate(W), np.concatenate(B)
    cw, cb = STAT(W), STAT(B)
    return cw, cb, cb - cw

def perm_p(blocks, n_perm=N_PERM, seed=0):
    rng = np.random.default_rng(seed)
    obs = pooled_stat(blocks)[2]
    null = np.array([pooled_stat(blocks, [rng.permutation(lab) for _, lab in blocks])[2]
                     for _ in range(n_perm)])
    return (1 + np.sum(null >= obs)) / (n_perm + 1)

def boot_ci(blocks, n_boot=N_BOOT, seed=1):
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for k in range(n_boot):
        W, B = [], []
        for Dsub, lab in blocks:
            idx = np.concatenate([rng.choice(np.where(lab == g)[0], size=(lab == g).sum(), replace=True)
                                  for g in np.unique(lab)])
            w, bt = within_between(Dsub[np.ix_(idx, idx)], lab[idx], ids=idx)
            W.append(w); B.append(bt)
        diffs[k] = STAT(np.concatenate(B)) - STAT(np.concatenate(W))
    return np.percentile(diffs, [2.5, 97.5])

def fmt_p(p, n_perm=N_PERM):
    if p <= 1 / (n_perm + 1) + 1e-12:
        return f"p < {1/(n_perm+1):.1e}"
    return f"p = {p:.3g}"

rows = []
for name, pr in pairs.items():
    blocks = pair_blocks(pr)
    cw, cb, diff = pooled_stat(blocks)
    lo, hi = boot_ci(blocks)
    rows.append(dict(comparison=name, within=cw, between=cb, diff=diff,
                     ci_low=lo, ci_high=hi, p_raw=perm_p(blocks)))
R = pd.DataFrame(rows)
R['p_holm'] = multipletests(R.p_raw, method='holm')[1]
print(f'=== 稳健性分析：{STAT_NAME}, pooled within-experiment distances ===')
for _, r in R.iterrows():
    print(f"{r['comparison']}: {STAT_NAME} within {r['within']:.2f} vs between {r['between']:.2f}; "
          f"diff {r['diff']:.2f}, 95% CI [{r['ci_low']:.2f}, {r['ci_high']:.2f}]; "
          f"{fmt_p(r['p_raw'])} (Holm {fmt_p(r['p_holm'])})")
R.to_csv(out_dir + f"wasserstein_stats_{STAT_NAME}_CI_{TAG}.csv", index=False)



In [ ]:
# [W6] 实验水平分析（n = 3，独立实验为单位，保守补充分析）
# 每次实验得到一个 log(between/within) 值；H0: 均值 = 0（组间 = 组内），单侧 H1: > 0
def exact_signflip_p(x):
    """n=3 的精确符号翻转检验（单侧）；n=3 时最小可能 p = 1/8 = 0.125"""
    x = np.asarray(x, float); obs = x.mean()
    signs = np.array(list(itertools.product([1, -1], repeat=len(x))))
    null = (signs * x).mean(axis=1)
    return np.mean(null >= obs - 1e-12)

rows3 = []
for k in names:
    lr = np.log(res[k]['ratios'])
    t = stats.ttest_1samp(lr, 0, alternative='greater')
    se = lr.std(ddof=1) / np.sqrt(len(lr))
    tcrit = stats.t.ppf(0.975, df=len(lr) - 1)
    pt = stats.ttest_rel(res[k]['between'], res[k]['within'], alternative='greater')   # 配对 t：between vs within 均值
    rows3.append(dict(
        comparison=k,
        ratio_e1=res[k]['ratios'][0], ratio_e2=res[k]['ratios'][1], ratio_e3=res[k]['ratios'][2],
        geo_mean_ratio=np.exp(lr.mean()),
        ratio_CI95_low=np.exp(lr.mean() - tcrit * se), ratio_CI95_high=np.exp(lr.mean() + tcrit * se),
        t_logratio=t.statistic, p_t_logratio=t.pvalue,
        p_paired_t=pt.pvalue,
        p_signflip_exact=exact_signflip_p(lr),
        consistent=f"{res[k]['n_pos']}/3"))
R3 = pd.DataFrame(rows3)
R3['p_t_logratio_holm'] = multipletests(R3.p_t_logratio, method='holm')[1]
R3['p_paired_t_holm']   = multipletests(R3.p_paired_t,   method='holm')[1]
print('=== 实验水平分析（n = 3 independent experiments）===')
print(R3.round(4).to_string(index=False))
R3.to_csv(out_dir + f"wasserstein_stats_n3_{TAG}.csv", index=False)



In [ ]:
# [W7] 汇总表：主分析 + 稳健性 + n=3
final = pd.DataFrame({
    'comparison': names,
    'ratio_e1/e2/e3': [np.round(res[k]['ratios'], 3) for k in names],
    'consistent': [f"{res[k]['n_pos']}/3" for k in names],
    'cell_perm_mean_p_holm': p_adj,
    f'cell_perm_{STAT_NAME}_p_holm': R.set_index('comparison').loc[names, 'p_holm'].values,
    f'{STAT_NAME}_diff_CI95': [f"{a:.2f} [{b:.2f}, {c:.2f}]" for a, b, c in
                              R.set_index('comparison').loc[names, ['diff', 'ci_low', 'ci_high']].values],
    'n3_geo_ratio_CI95': [f"{a:.3f} [{b:.3f}, {c:.3f}]" for a, b, c in
                          R3.set_index('comparison').loc[names, ['geo_mean_ratio', 'ratio_CI95_low', 'ratio_CI95_high']].values],
    'n3_t_logratio_p_holm': R3.set_index('comparison').loc[names, 'p_t_logratio_holm'].values,
})
print('=== 汇总 ===')
print(final.to_string(index=False))
final.to_csv(out_dir + f"wasserstein_stats_summary_{TAG}.csv", index=False)
